# מעבדה 05 — עבודה ומסלולים תרמודינמיים

במעבדה הזו תבנו בעצמכם מסלולים במישור $P$–$V$, תמדדו את העבודה לאורך כל אחד מהם, ותצפו
בפונקציית מצב ובפונקציית מסלול מתנהגות באופן שונה מול עיניכם.

כל המודול נשען על השוואה אחת, ולכן אל תמהרו: שני מסלולים, נקודות קצה זהות, עבודה שונה.

## מפרט המודל

| | |
|---|---|
| **מערכת** | כמות קבועה של גז אידיאלי בגליל הסגור בבוכנה |
| **דינמיקה** | כמו-סטטית — הגז בשיווי משקל בכל נקודה, ולכן זוג יחיד $(P,V)$ מתאר אותו |
| **גבול** | בוכנה נטולת חיכוך; חום רשאי לחצות אלא אם התהליך אדיאבטי |
| **צבר** | אינו רלוונטי — תרמודינמיקה מאקרוסקופית, ללא ספירת מיקרו-מצבים |
| **מוזנח** | חיכוך, מערבולות, אפקטים של קצב סופי, אי-אידיאליות, מסת הבוכנה |
| **תקף כאשר** | התהליך איטי ביחס לזמן הרלקסציה הפנימי של הגז |
| **אופני כישלון** | תהליכים מהירים או בלתי הפיכים, שבהם לגז אין לחץ יחיד והשטח מתחת לעקומה *אינו* העבודה |

מוסכמת הסימנים לאורך כל הדרך, ללא יוצא מן הכלל:

$$ dU = \delta Q + \delta W_{\mathrm{on}}, \qquad \delta W_{\mathrm{on}} = -P\,dV. $$


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from thermolab import paths
from thermolab.validation import convergence_study, relative_error

N_PARTICLES = 1000
TEMPERATURE = 300.0  # K
V1, V2 = 1.0e-3, 2.0e-3  # m^3
GAMMA = 5.0 / 3.0  # monatomic ideal gas

P1 = paths.ideal_gas_pressure(N_PARTICLES, TEMPERATURE, V1)
P2 = paths.ideal_gas_pressure(N_PARTICLES, TEMPERATURE, V2)

print(f"start  V = {V1 * 1e3:.2f} L   P = {P1:.4e} Pa")
print(f"finish V = {V2 * 1e3:.2f} L   P = {P2:.4e} Pa")
print(f"both at T = {TEMPERATURE} K")

### לנבא

שלושה מסלולים מעבירים את הגז ממצב ההתחלה למצב הסיום:

- **איזותרמה** — התפשטות בטמפרטורה קבועה.
- **איזובר ואז איזוכור** — התפשטות בלחץ ההתחלתי, ואז קירור בנפח הסופי.
- **איזוכור ואז איזובר** — קירור בנפח ההתחלתי, ואז התפשטות בלחץ הסופי.

לפני שתריצו משהו, דרגו אותם לפי גודל העבודה הנעשית על הגז, ונבאו אם $\Delta U$ שונה
ביניהם.


**הניבוי שלכם:**

*(כתבו כאן לפני הרצת התא הבא)*


In [ ]:
isotherm = paths.isothermal_path(N_PARTICLES, TEMPERATURE, V1, V2, label="isotherm")

isobar_first = paths.join(
    paths.isobaric_path(P1, V1, V2),
    paths.isochoric_path(V2, P1, P2),
    label="isobar then isochore",
)

isochore_first = paths.join(
    paths.isochoric_path(V1, P1, P2),
    paths.isobaric_path(P2, V1, V2),
    label="isochore then isobar",
)

routes = [isotherm, isobar_first, isochore_first]

print(f"{'route':<24}{'W_on (J)':>14}{'dU (J)':>14}{'Q (J)':>14}")
for route in routes:
    print(
        f"{route.label:<24}{route.work_on_gas():>14.4e}"
        f"{route.internal_energy_change():>14.4e}{route.heat_into_gas():>14.4e}"
    )

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
colours = ["#2563eb", "#d97706", "#059669"]

for route, colour in zip(routes, colours, strict=True):
    ax.plot(route.volumes * 1e3, route.pressures, color=colour, lw=2, label=route.label)
    ax.fill_between(route.volumes * 1e3, route.pressures, alpha=0.10, color=colour)

ax.plot([V1 * 1e3, V2 * 1e3], [P1, P2], "ko", ms=8, zorder=5)
ax.annotate("A", (V1 * 1e3, P1), textcoords="offset points", xytext=(-16, 6), fontsize=13)
ax.annotate("B", (V2 * 1e3, P2), textcoords="offset points", xytext=(8, -14), fontsize=13)
ax.set_xlabel("volume (L)")
ax.set_ylabel("pressure (Pa)")
ax.set_title("Same endpoints, different areas underneath")
ax.legend()
plt.tight_layout()
plt.show()

השטחים המוצללים הם העבודות, והם שונים באופן נראה לעין. השינויים באנרגיה הפנימית אינם רק
קרובים — הם זהים עד לדיוק המכונה, וכולם אפס, משום ששתי נקודות הקצה יושבות על אותה איזותרמה
והאנרגיה הפנימית של גז אידיאלי תלויה רק בטמפרטורה.

שימו לב למה שהחוק הראשון מאלץ אז: מכיוון ש-$\Delta U$ מקובע ו-$W_{\mathrm{on}}$ אינו,
$Q$ חייב להיות שונה בדיוק בכמות המקזזת. שתי פונקציות המסלול חופשיות; רק סכומן אינו.


In [ ]:
works = np.array([route.work_on_gas() for route in routes])
energies = np.array([route.internal_energy_change() for route in routes])

print(f"work values (J): {works}")
print(f"fractional spread of W_on: {(works.max() - works.min()) / abs(works.mean()):.3f}")
print(f"absolute spread of dU (J): {energies.max() - energies.min():.3e}")

# A trap worth seeing once. These energies are around 1e-18 J, far inside numpy's default
# absolute tolerance, so a careless comparison "proves" that work is a state function.
print(f"\nnp.isclose on two different works -> {np.isclose(works[0], works[1])}  (misleading!)")
print(f"fractional difference             -> {relative_error(works[0], works[1]):.3f}  (honest)")

## חלק 2 — סיבוב במעגל

מחזור מחזיר את הגז בדיוק למצבו ההתחלתי. פונקציית המצב חייבת לחזור לערכה; פונקציות המסלול
אינן חייבות. כל מה שמנוע עושה חי בפער הזה.


In [ ]:
cycle = paths.join(
    paths.isobaric_path(P1, V1, V2),
    paths.isochoric_path(V2, P1, P2),
    paths.isobaric_path(P2, V2, V1),
    paths.isochoric_path(V1, P2, P1),
    label="rectangular cycle",
)

enclosed_area = (P1 - P2) * (V2 - V1)

print(f"closed?                {cycle.is_closed}")
print(f"net W_on per cycle     {cycle.work_on_gas():.4e} J")
print(f"minus enclosed area    {-enclosed_area:.4e} J")
print(f"net dU per cycle       {cycle.internal_energy_change():.4e} J")
print(f"net Q per cycle        {cycle.heat_into_gas():.4e} J")

fig, ax = plt.subplots(figsize=(6, 4.5))
ax.plot(cycle.volumes * 1e3, cycle.pressures, color="#7c3aed", lw=2)
ax.fill(cycle.volumes * 1e3, cycle.pressures, alpha=0.15, color="#7c3aed")
ax.set_xlabel("volume (L)"), ax.set_ylabel("pressure (Pa)")
ax.set_title("net work = enclosed area")
plt.tight_layout()
plt.show()

## חלק 3 — איזותרמה מול אדיאבטה

שתי התפשטויות מאותה נקודת מוצא לאותו נפח סופי. לאורך האיזותרמה זורם חום פנימה כדי להחזיק את
הטמפרטורה למעלה; לאורך האדיאבטה לא זורם חום כלל, ולכן הגז מתקרר תוך שהוא מוציא מאנרגייתו
הפנימית. האדיאבטה תלולה יותר, והיא מבצעת פחות עבודה.


In [ ]:
adiabat = paths.adiabatic_path(P1, V1, V2, GAMMA, label="adiabat")
p_end_adiabatic = adiabat.end[1]

exact_isothermal = paths.isothermal_work_on_gas(N_PARTICLES, TEMPERATURE, V1, V2)
exact_adiabatic = paths.adiabatic_work_on_gas(P1, V1, p_end_adiabatic, V2, GAMMA)

print(f"isotherm  W_on = {isotherm.work_on_gas():.6e} J   closed form {exact_isothermal:.6e} J")
print(f"adiabat   W_on = {adiabat.work_on_gas():.6e} J   closed form {exact_adiabatic:.6e} J")
print(f"adiabat   Q    = {adiabat.heat_into_gas():.3e} J  (should be ~0 relative to W)")
print(
    f"\nfinal temperature: isotherm {TEMPERATURE:.1f} K, "
    f"adiabat {paths.ideal_gas_temperature(N_PARTICLES, p_end_adiabatic, V2):.1f} K"
)

fig, ax = plt.subplots(figsize=(6.5, 4.5))
ax.plot(isotherm.volumes * 1e3, isotherm.pressures, lw=2, label="isotherm")
ax.plot(adiabat.volumes * 1e3, adiabat.pressures, lw=2, label="adiabat")
ax.set_xlabel("volume (L)"), ax.set_ylabel("pressure (Pa)"), ax.legend()
ax.set_title("the adiabat falls faster")
plt.tight_layout()
plt.show()

## חלק 4 — עורך המסלולים

כעת בנו מסלול משלכם. המחוונים מזיזים שתי נקודות בקרה ביניים; המחברת מחברת אותן למסלול קטעי,
מצללת את השטח, ומדווחת את העבודה לצד הייחוס האיזותרמי בין אותן נקודות קצה.

שני דברים לנסות:

1. מצאו מסלול המבצע *יותר* עבודה על הסביבה מן המסלול האיזוברי.
2. מצאו שני מסלולים שונים לעין שהעבודה שלהם כמעט שווה, והסבירו מה משותף להם.


In [ ]:
import ipywidgets as widgets


def edit_path(p_mid1_frac=1.0, v_mid1_frac=0.33, p_mid2_frac=0.6, v_mid2_frac=0.66):
    """Piecewise-linear route A -> C1 -> C2 -> B, with the control points set by sliders."""
    volumes = np.array(
        [
            V1,
            V1 + v_mid1_frac * (V2 - V1),
            V1 + v_mid2_frac * (V2 - V1),
            V2,
        ]
    )
    pressures = np.array([P1, p_mid1_frac * P1, p_mid2_frac * P1, P2])

    # Sample each straight leg finely so the trapezoid rule is exact on it.
    dense_v, dense_p = [], []
    for leg in range(len(volumes) - 1):
        va, vb = volumes[leg], volumes[leg + 1]
        pa, pb = pressures[leg], pressures[leg + 1]
        leg_v = np.linspace(va, vb, 200)
        leg_p = np.interp(leg_v, [va, vb], [pa, pb]) if vb != va else np.linspace(pa, pb, 200)
        dense_v.append(leg_v)
        dense_p.append(leg_p)
    route = paths.Path(np.concatenate(dense_v), np.concatenate(dense_p), "your route")

    fig, ax = plt.subplots(figsize=(7, 4.5))
    ax.plot(isotherm.volumes * 1e3, isotherm.pressures, "--", color="grey", label="isotherm")
    ax.plot(route.volumes * 1e3, route.pressures, color="#2563eb", lw=2, label="your route")
    ax.fill_between(route.volumes * 1e3, route.pressures, alpha=0.15, color="#2563eb")
    ax.plot(volumes * 1e3, pressures, "o", color="#1e3a8a")
    ax.set_xlabel("volume (L)"), ax.set_ylabel("pressure (Pa)"), ax.legend()
    ax.set_title(
        f"W_on = {route.work_on_gas():.4e} J     "
        f"isotherm = {isotherm.work_on_gas():.4e} J     "
        f"dU = {route.internal_energy_change():.2e} J"
    )
    plt.tight_layout()
    plt.show()


widgets.interact(
    edit_path,
    p_mid1_frac=widgets.FloatSlider(min=0.3, max=1.4, step=0.05, value=1.0, description="P at C1"),
    v_mid1_frac=widgets.FloatSlider(
        min=0.05, max=0.9, step=0.05, value=0.33, description="V at C1"
    ),
    p_mid2_frac=widgets.FloatSlider(min=0.3, max=1.4, step=0.05, value=0.6, description="P at C2"),
    v_mid2_frac=widgets.FloatSlider(
        min=0.1, max=0.95, step=0.05, value=0.66, description="V at C2"
    ),
);

## חלק 5 — בדיקות אוטומטיות

אותן טענות רצות בחבילת הבדיקות של הפרויקט, כך שהטענות בעמוד המודול אינן יכולות להירקב
בשקט.


In [ ]:
# 1. Closed forms. These paths are sampled on 257 points for drawing, so the trapezoid rule
#    leaves a residual of order (1/257)^2 — small, but not machine precision. Check 5 refines
#    the sampling deliberately and watches the error fall.
iso_error = relative_error(isotherm.work_on_gas(), exact_isothermal)
adi_error = relative_error(adiabat.work_on_gas(), exact_adiabatic)
assert iso_error < 1e-4
assert adi_error < 1e-4

# 2. Path dependence, compared fractionally rather than absolutely.
assert relative_error(isotherm.work_on_gas(), isobar_first.work_on_gas()) > 0.25

# 3. State function agreement to machine precision.
assert abs(isotherm.internal_energy_change() - isobar_first.internal_energy_change()) < 1e-30

# 4. The first law closes on every route.
for route in routes + [adiabat, cycle]:
    residual = route.heat_into_gas() + route.work_on_gas() - route.internal_energy_change()
    assert abs(residual) < 1e-24, route.label

# 5. Quadrature convergence: the trapezoid rule is second order, and we measure it.
study = convergence_study(
    lambda points: paths.work_along(
        lambda v: paths.isothermal_pressure(v, N_PARTICLES, TEMPERATURE), V1, V2, n_points=points
    ),
    refinements=[17, 33, 65, 129, 257],
    exact=exact_isothermal,
)
assert abs(study.observed_order - 2.0) < 0.2

print(f"isotherm quadrature error at 257 points: {iso_error:.2e}")
print(f"adiabat  quadrature error at 257 points: {adi_error:.2e}")
print(f"observed convergence order:              {study.observed_order:.3f}  (theory: 2)")
print("\nall checks passed")

In [ ]:
plt.figure(figsize=(5.5, 4))
plt.loglog(study.refinements, study.errors, "o-", label="measured error")
plt.loglog(
    study.refinements,
    study.errors[0] * (study.refinements / study.refinements[0]) ** -2.0,
    "--",
    label=r"$n^{-2}$",
)
plt.xlabel("sample points"), plt.ylabel("relative error"), plt.legend()
plt.title("trapezoid rule converges at second order")
plt.tight_layout()
plt.show()

## בדקו את הבנתכם

הריצו את התא שלהלן לחידון עם בדיקה אוטומטית.


In [ ]:
import json
from pathlib import Path

quiz_path = Path("..") / "_quiz" / "05-work-paths.json"
if quiz_path.exists():
    from jupyterquiz import display_quiz

    # Parsed here with an explicit encoding: jupyterquiz opens the file with the platform
    # default, which cannot decode the Hebrew edition of this notebook on Windows.
    display_quiz(json.loads(quiz_path.read_text(encoding="utf-8")))
else:
    print("Quiz not generated yet — run: uv run python scripts/render_quizzes.py")

## לפני שאתם עוזבים

1. זה עתה חישבתם שלושה ערכים שונים של $W_{\mathrm{on}}$ בין זוג מצבים אחד. הסבירו במילים
   שלכם מדוע הדבר אינו מפר את שימור האנרגיה.
2. כל המסלולים במחברת הזו היו כמו-סטטיים. נקבו בגודל אחד שחושב כאן שהיה מאבד את משמעותו
   בדחיסה פתאומית, ואמרו בדיוק מדוע.
3. ההשוואה עם `np.isclose` שלמעלה החזירה תשובה מטעה. איזה לקח כללי על בדיקת פיזיקה נומרית
   נושא הדבר?


**התשובות שלכם:**

1.
2.
3.
